HF token

In [ ]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Use secret if running in Google Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Store Hugging Face data under `/content` if running in Colab Enterprise
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Authenticate with Hugging Face
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

# connect to the google drive

In [ ]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image
from datasets import Dataset

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Dissertation"
Mendeley_path = PROJECT_DIR + "/datasets/Bone_fracture_dataset/Original"

Mounted at /content/drive


Install dependencies

In [ ]:
! pip install --upgrade --quiet bitsandbytes datasets evaluate peft tensorboard transformers trl torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 144.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires p

Load dataset

Extract the files

In [ ]:
import zipfile, os

base = "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX"
parts = ["images_part2", "images_part3", "images_part4"]

for part in parts:
    zip_path = f"{base}/{part}.zip"
    extract_path = f"/content/{part}"
    if not os.path.exists(extract_path):
        print(f"Extracting {part}...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(extract_path)
        print(f"Done: {part}, files: {len(os.listdir(extract_path))}")
    else:
        print(f"{part} already exists, files: {len(os.listdir(extract_path))}")

Extracting images_part2...
Done: images_part2, files: 5143
Extracting images_part3...
Done: images_part3, files: 4842
Extracting images_part4...
Done: images_part4, files: 5311


In [ ]:
zip_path = "/content/drive/MyDrive/Dissertation/datasets/GRAZPEDWRI-DX/folder_structure.zip"
extract_to = "/content/yolov5_labels"

if not os.path.exists(f"{extract_to}/yolov5/labels"):
    with zipfile.ZipFile(zip_path) as z:
        members = [n for n in z.namelist() if n.startswith("yolov5/")]
        z.extractall(path=extract_to, members=members)
    print("Extracted yolov5 labels")
else:
    print("yolov5 labels already present")

Extracted yolov5 labels


In [ ]:
import pandas as pd
import os

PROJECT_DIR = "/content/drive/MyDrive/Dissertation"
Graz_path = PROJECT_DIR + "/datasets/GRAZPEDWRI-DX"

df = pd.read_csv(Graz_path + "/dataset.csv")

# Match image paths across all 4 parts
parts = ["images_part1", "images_part2", "images_part3", "images_part4"]
path_map = {}
for part in parts:
    if part == "images_part1":
        dir_path = f"{Graz_path}/images_part1"
    else:
        dir_path = f"/content/{part}"
    for fname in os.listdir(dir_path):
        stem = os.path.splitext(fname)[0]
        path_map[stem] = os.path.join(dir_path, fname)

df["image_path"] = df["filestem"].map(path_map)
df_matched = df.dropna(subset=["image_path"]).copy()
print(f"Matched: {len(df_matched)}")

Matched: 20327


location of the fracture

In [ ]:
def yolo_box_to_location(x_center, y_center):
    vertical = "distal" if y_center < 0.5 else "proximal"
    horizontal = "radial" if x_center < 0.5 else "ulnar"
    return f"{vertical} {horizontal} region"

def get_fracture_locations(filestem, labels_dir="/content/yolov5_labels/yolov5/labels"):
    path = os.path.join(labels_dir, f"{filestem}.txt")
    if not os.path.exists(path):
        return []
    locations = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if parts and int(parts[0]) == 3:
                x_center, y_center = float(parts[1]), float(parts[2])
                locations.append(yolo_box_to_location(x_center, y_center))
    return locations

In [ ]:
PROMPT_B = "Is there a fracture visible in this pediatric wrist X-ray?\nA: Yes, fracture present\nB: No fracture present"

FRACTURE_CLASSES_B = ["A: Yes, fracture present", "B: No fracture present"]

def row_to_example_b(row):
    has_fracture = pd.notna(row['fracture_visible']) and row['fracture_visible'] == 1.0
    label = 0 if has_fracture else 1  # index into FRACTURE_CLASSES_B

    # location detail as extra context appended after the label,
    # so the model still learns it, but the core answer stays constrained
    extra = ""
    if has_fracture:
        locations = get_fracture_locations(row["filestem"])
        if locations:
            loc_text = " and ".join(set(locations))
            extra = f" Location: {loc_text}."

    return {
        "image_path": row["image_path"],
        "label": label,
        "extra_detail": extra,
    }

examples_b = df_matched.apply(row_to_example_b, axis=1).tolist()
print(f"Total: {len(examples_b)}")

Total: 20327


Build dataset, then split

In [ ]:
from datasets import Dataset, Features, ClassLabel, Value

features_b = Features({
    "image_path": Value("string"),
    "label": ClassLabel(names=FRACTURE_CLASSES_B),
    "extra_detail": Value("string"),
})

graz_ds = Dataset.from_list(examples_b, features=features_b)

split1 = graz_ds.train_test_split(test_size=0.2, stratify_by_column="label", seed=42)
train_ds_b_full = split1["train"]
temp_ds_b = split1["test"]

split2 = temp_ds_b.train_test_split(test_size=0.5, stratify_by_column="label", seed=42)
val_ds_b = split2["train"]
test_ds_b = split2["test"]

print("Train:", train_ds_b_full.num_rows, "Val:", val_ds_b.num_rows, "Test:", test_ds_b.num_rows)

Train: 16261 Val: 2033 Test: 2033


Balance training set

In [ ]:
def format_data_b(example):
    response = FRACTURE_CLASSES_B[example["label"]] + example.get("extra_detail", "")
    example["messages"] = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT_B}]},
        {"role": "assistant", "content": [{"type": "text", "text": response}]},
    ]
    return example
train_ds_b = train_ds_b_full.map(format_data_b)
val_ds_b = val_ds_b.map(format_data_b)
test_ds_b = test_ds_b.map(format_data_b)


Map:   0%|          | 0/16261 [00:00<?, ? examples/s]

Map:   0%|          | 0/2033 [00:00<?, ? examples/s]

Map:   0%|          | 0/2033 [00:00<?, ? examples/s]

In [ ]:
train_df_b_v2 = train_ds_b_full.to_pandas()
fx_v2 = train_df_b_v2[train_df_b_v2["label"] == 0]
no_fx_v2 = train_df_b_v2[train_df_b_v2["label"] == 1]

target_ratio_v2 = 1.2  # tighter than before (was 1.5)
target_n_v2 = int(len(no_fx_v2) * target_ratio_v2)
fx_balanced_v2 = fx_v2.sample(n=min(target_n_v2, len(fx_v2)), random_state=42)

train_df_b2_balanced = pd.concat([fx_balanced_v2, no_fx_v2]).sample(frac=1, random_state=42).reset_index(drop=True)
print(train_df_b2_balanced["label"].value_counts())

from datasets import Dataset
train_ds_b2 = Dataset.from_pandas(train_df_b2_balanced, features=features_b)
train_ds_b2 = train_ds_b2.map(format_data_b)

label
0    6505
1    5421
Name: count, dtype: int64


Map:   0%|          | 0/11926 [00:00<?, ? examples/s]

Load model

In [ ]:
import gc, torch
for var in ["trainer_b", "model", "base_model"]:
    try:
        exec(f"del {var}")
    except NameError:
        pass
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-4b-it"
model = AutoModelForImageTextToText.from_pretrained(
    model_id, attn_implementation="sdpa", torch_dtype=torch.bfloat16, device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

LoRA config

In [ ]:
from peft import LoraConfig

peft_config_v2 = LoraConfig(
    lora_alpha=32,       # up from 16, matching r
    lora_dropout=0.05,
    r=32,                # up from 16
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
)

Collate function

In [ ]:
from typing import Any
from PIL import Image

def collate_fn(examples: list[dict[str, Any]]):
    texts = []
    images = []
    for example in examples:
        image = Image.open(example["image_path"]).convert("RGB")
        images.append([image])
        texts.append(processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        ).strip())

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()

    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch

Training Config

In [ ]:
from trl import SFTConfig

args_b2 = SFTConfig(
    output_dir="medgemma-4b-it-sft-lora-grazpedwri-v2",
    num_train_epochs=3,   # up from 2
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=50,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=2e-4,
    bf16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="linear",
    push_to_hub=False,
    report_to="tensorboard",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
)

Train the model





In [ ]:
from trl import SFTTrainer

trainer_b2 = SFTTrainer(
    model=model,
    args=args_b2,
    train_dataset=train_ds_b2,
    eval_dataset=val_ds_b.shuffle(seed=42).select(range(200)),
    peft_config=peft_config_v2,
    processing_class=processor,
    data_collator=collate_fn,
)

trainer_b2.train()

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.513518,0.072563,0.067706,244718.000000,0.951719
100,0.068623,0.071686,0.062014,489602.000000,0.957125
150,0.069932,0.067600,0.073947,734405.000000,0.957796
200,0.068548,0.071725,0.066239,979594.000000,0.949508
250,0.068042,0.068033,0.068744,1224614.000000,0.956481
300,0.066155,0.069158,0.064808,1469553.000000,0.953164
350,0.066285,0.067450,0.067863,1714505.000000,0.954118
400,0.065979,0.067903,0.065228,1959517.000000,0.957796
450,0.065270,0.068896,0.065075,2204374.000000,0.956670
500,0.065282,0.067817,0.067252,2449198.000000,0.955580


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.513518,0.072563,0.067706,244718.000000,0.951719
100,0.068623,0.071686,0.062014,489602.000000,0.957125
150,0.069932,0.067600,0.073947,734405.000000,0.957796
200,0.068548,0.071725,0.066239,979594.000000,0.949508
250,0.068042,0.068033,0.068744,1224614.000000,0.956481
300,0.066155,0.069158,0.064808,1469553.000000,0.953164
350,0.066285,0.067450,0.067863,1714505.000000,0.954118
400,0.065979,0.067903,0.065228,1959517.000000,0.957796
450,0.065270,0.068896,0.065075,2204374.000000,0.956670
500,0.065282,0.067817,0.067252,2449198.000000,0.955580


TrainOutput(global_step=2238, training_loss=0.062494696624796765, metrics={'train_runtime': 20778.4387, 'train_samples_per_second': 1.722, 'train_steps_per_second': 0.108, 'total_flos': 2.9224347897222374e+17, 'train_loss': 0.062494696624796765, 'epoch': 3.0})

Save model

In [ ]:
trainer_b2.save_model("medgemma-4b-it-sft-lora-grazpedwri-final-second")
processor.save_pretrained("medgemma-4b-it-sft-lora-grazpedwri-final-second")

!mkdir -p /content/drive/MyDrive/Dissertation/models
!cp -r medgemma-4b-it-sft-lora-grazpedwri-final-second /content/drive/MyDrive/Dissertation/models/
print("Saved")

Saved


Check loss curve

In [ ]:
print(trainer_b2.state.log_history[-10:])

[{'loss': 0.036583261489868166, 'grad_norm': 0.5559002161026001, 'learning_rate': 1.6890080428954423e-05, 'entropy': 0.035171826211735605, 'num_tokens': 10035427.0, 'mean_token_accuracy': 0.9787810152769089, 'epoch': 2.7484909456740443, 'step': 2050}, {'eval_loss': 0.15668754279613495, 'eval_runtime': 25.8714, 'eval_samples_per_second': 7.731, 'eval_steps_per_second': 1.933, 'eval_entropy': 0.03747979525476694, 'eval_num_tokens': 10035427.0, 'eval_mean_token_accuracy': 0.956306802034378, 'epoch': 2.7484909456740443, 'step': 2050}, {'loss': 0.035405631065368655, 'grad_norm': 0.6299988031387329, 'learning_rate': 1.2421805183199285e-05, 'entropy': 0.035149388303980234, 'num_tokens': 10280398.0, 'mean_token_accuracy': 0.9796339645981789, 'epoch': 2.8155600268276326, 'step': 2100}, {'eval_loss': 0.16112969815731049, 'eval_runtime': 25.8163, 'eval_samples_per_second': 7.747, 'eval_steps_per_second': 1.937, 'eval_entropy': 0.03768357120454311, 'eval_num_tokens': 10280398.0, 'eval_mean_token_a

Compute metrics

In [ ]:
def postprocess_b(response_text, do_full_match=False):
    if do_full_match:
        try:
            return FRACTURE_CLASSES_B.index(response_text.strip())
        except ValueError:
            # allow for the trailing "Location: ..." text after the label
            for i, label in enumerate(FRACTURE_CLASSES_B):
                if response_text.strip().startswith(label):
                    return i
            return None
    for i, label in enumerate(FRACTURE_CLASSES_B):
        if label in response_text:
            return i
    return None

In [ ]:
def generate_response(image_path, prompt, model, processor, max_new_tokens=30):
    image = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Check if model is alive

In [ ]:
print("model" in dir())
print("trainer_b2" in dir())

False
False


Reload model from drive

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
import torch

base_model_id = "google/medgemma-4b-it"
b2_path = "/content/drive/MyDrive/Dissertation/models/medgemma-4b-it-sft-lora-grazpedwri-final-second"

base_for_b2 = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_for_b2, b2_path)
processor = AutoProcessor.from_pretrained(b2_path)
processor.tokenizer.padding_side = "right"

print("Loaded Model B2 successfully")

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Loaded Model B2 successfully


In [ ]:
import pandas as pd

results_b2 = []
for i, example in enumerate(test_ds_b):
    gen_text = generate_response(example["image_path"], PROMPT_B, model, processor, max_new_tokens=30)
    pred = postprocess_b(gen_text)
    results_b2.append({
        "ground_truth": example["label"],
        "prediction": pred,
        "generated_text": gen_text,
    })
    if i % 100 == 0:
        print(f"{i}/{len(test_ds_b)} done")

results_b2_df = pd.DataFrame(results_b2)
results_b2_df.to_csv("/content/drive/MyDrive/Dissertation/results/model_b2_results.csv", index=False)

0/2033 done
100/2033 done
200/2033 done
300/2033 done
400/2033 done
500/2033 done
600/2033 done
700/2033 done
800/2033 done
900/2033 done
1000/2033 done
1100/2033 done
1200/2033 done
1300/2033 done
1400/2033 done
1500/2033 done
1600/2033 done
1700/2033 done
1800/2033 done
1900/2033 done
2000/2033 done


Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

valid_b2 = results_b2_df.dropna(subset=["prediction"])
print(f"Unparseable: {len(results_b2_df) - len(valid_b2)} / {len(results_b2_df)}")
print(f"Accuracy: {accuracy_score(valid_b2['ground_truth'], valid_b2['prediction']):.3f}")
print(f"Precision: {precision_score(valid_b2['ground_truth'], valid_b2['prediction'], pos_label=0, zero_division=0):.3f}")
print(f"Recall: {recall_score(valid_b2['ground_truth'], valid_b2['prediction'], pos_label=0, zero_division=0):.3f}")
print(f"F1 (weighted): {f1_score(valid_b2['ground_truth'], valid_b2['prediction'], average='weighted'):.3f}")
print(confusion_matrix(valid_b2['ground_truth'], valid_b2['prediction']))

Unparseable: 0 / 2033
Accuracy: 0.558
Precision: 0.763
Recall: 0.488
F1 (weighted): 0.568
[[661 694]
 [205 473]]
